# SUPERSTORE DATA ANALYSIS — PART 1: DATA CLEANING AND PREPARATION


# IMPORTING LIBRARIES


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime



# LOADING DATASET


In [2]:
sales_data = pd.read_csv("/content/Superstore.csv")
sales_data.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


# BASIC DATA UNDERSTANDING


In [3]:
sales_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [4]:
sales_data.shape


(9994, 21)

In [5]:
sales_data.columns


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

# MISSING VALUES


In [6]:
sales_data.isnull().sum()


,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [7]:
# finding which rows has null values
missing_postal = sales_data[sales_data["Postal Code"].isnull()]
missing_postal[["City", "State", "Region"]].drop_duplicates()


,City,State,Region
2234,Burlington,Vermont,East


In [8]:
# filling postal code with 05401 since all missing postal codes belong to Burlington, Vermont
sales_data["Postal Code"] = sales_data["Postal Code"].fillna(5401)
sales_data.isnull().sum()


,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [9]:
# converting postal code into 5-digit string format
sales_data["Postal Code"] = sales_data["Postal Code"].astype(int).astype(str).str.zfill(5)
sales_data["Postal Code"].head()


,Postal Code
0,42420
1,42420
2,90036
3,33311
4,33311


In [10]:
# converting dates to datetime format
sales_data["Order Date"] = pd.to_datetime(sales_data["Order Date"])
sales_data["Ship Date"] = pd.to_datetime(sales_data["Ship Date"])

sales_data[["Order Date", "Ship Date"]].dtypes


,0
Order Date,datetime64[ns]
Ship Date,datetime64[ns]


In [11]:
# creating columns for year, month, quarter and day of week for further analysis
sales_data["Year"] = sales_data["Order Date"].dt.year
sales_data["Month"] = sales_data["Order Date"].dt.to_period("M").astype(str)
sales_data["Month_Num"] = sales_data["Order Date"].dt.month
sales_data["Month_Name"] = sales_data["Order Date"].dt.strftime("%b")
sales_data["Quarter"] = "Q" + sales_data["Order Date"].dt.quarter.astype(str)
sales_data["DayOfWeek"] = sales_data["Order Date"].dt.dayofweek
sales_data["DayName"] = sales_data["Order Date"].dt.day_name()
sales_data["Shipping Days"] = (sales_data["Ship Date"] - sales_data["Order Date"]).dt.days
sales_data["Profit Margin (%)"] = ((sales_data["Profit"] / sales_data["Sales"]) * 100).round(2)
sales_data["Unit Price"] = (sales_data["Sales"] / sales_data["Quantity"]).round(2)
sales_data["COGS"] = (sales_data["Sales"] - sales_data["Profit"]).round(2)
sales_data["Is Loss"] = sales_data["Profit"] < 0


**Findings:**
1. Schema Expansion: Initial 21 raw columns expanded to 33 attributes after engineering operational and temporal metrics.
2. Missing Value Resolution: 11 null values in Postal Code (all originating from Burlington, Vermont) were imputed with standard USPS ZIP code `05401`.
3. Datetime Parsing: Order Date and Ship Date converted to native datetime format.
4. Engineered Features: Added Year, Month, Quarter, DayName, Shipping Days, Profit Margin (%), Unit Price, COGS, and Is Loss.


**Milestone 1**: Data schema is validated, missing values are resolved with zero data loss, and features are engineered for downstream analysis.


# DUPLICATE VALUES


In [12]:
sales_data.duplicated().sum()


np.int64(0)

In [13]:
duplicates = sales_data[sales_data.duplicated(keep=False)]
duplicates


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Month_Num,Month_Name,Quarter,DayOfWeek,DayName,Shipping Days,Profit Margin (%),Unit Price,COGS,Is Loss


In [14]:
sales_data = sales_data.drop_duplicates()
sales_data.duplicated().sum()


np.int64(0)

**Milestone 2**: Dataset verified free from duplicate records.


# DESCRIPTIVE STATISTICS


In [15]:
sales_data.describe().round(2)


,Row ID,Order Date,Ship Date,Sales,Quantity,Discount,Profit,Year,Month_Num,DayOfWeek,Shipping Days,Profit Margin (%),Unit Price,COGS
count,9994.00,9994,9994,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00
mean,4997.50,2017-04-30 05:17:08.056834048,2017-05-04 04:17:20.304182528,229.86,3.79,0.16,28.66,2016.72,7.81,2.99,3.96,12.03,60.92,201.20
min,1.00,2015-01-03 00:00:00,2015-01-07 00:00:00,0.44,1.00,0.00,-6599.98,2015.00,1.00,0.00,0.00,-275.00,0.34,0.55
25%,2499.25,2016-05-23 00:00:00,2016-05-27 00:00:00,17.28,2.00,0.00,1.73,2016.00,5.00,1.00,3.00,7.50,5.47,12.69
50%,4997.50,2017-06-26 00:00:00,2017-06-29 00:00:00,54.49,3.00,0.20,8.67,2017.00,9.00,3.00,4.00,27.00,16.27,41.66
75%,7495.75,2018-05-14 00:00:00,2018-05-18 00:00:00,209.94,5.00,0.20,29.36,2018.00,11.00,5.00,5.00,36.25,63.94,182.23
max,9994.00,2018-12-30 00:00:00,2019-01-05 00:00:00,22638.48,14.00,0.80,8399.98,2018.00,12.00,6.00,7.00,50.00,3773.08,24449.56
std,2885.16,NaN,NaN,623.25,2.23,0.21,234.26,1.12,3.28,2.18,1.75,46.68,142.93,550.84


In [16]:
# Exporting cleaned dataset for Part 2
sales_data.to_csv("cleaned_superstore.csv", index=False)
print(f"Cleaned dataset exported to 'cleaned_superstore.csv' with {len(sales_data):,} rows and {len(sales_data.columns)} columns.")


Cleaned dataset exported to 'cleaned_superstore.csv' with 9,994 rows and 33 columns.
